In [ ]:
# 1. Mount Drive. Program files stay local; your data persists on Drive.
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys
import time
import shutil

DRIVE_ROOT = '/content/drive/MyDrive/SwarmUI'
APP_ROOT = '/content/SwarmUI'
url = 'https://github.com/StMoelter/SwarmUI'
branch = 'fix/colab-existing-packages'

drive.mount('/content/drive')
if not Path('/content/drive').is_mount() or not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Google Drive is not mounted. Mount it before continuing.')

In [ ]:
# 2. Prepare local tools and checkout. Run all cells in order.
%cd /content
tools_started = time.monotonic()

# Match the .NET version checked by SwarmUI's launcher.
dotnet = Path.home() / '.dotnet/dotnet'
sdk_info = subprocess.run([str(dotnet), '--list-sdks'], capture_output=True, text=True) if dotnet.is_file() else None
if sdk_info is None or sdk_info.returncode != 0 or not any(line.startswith('10.0.') for line in sdk_info.stdout.splitlines()):
    !wget https://dot.net/v1/dotnet-install.sh -O dotnet-install.sh
    !chmod +x dotnet-install.sh
    !./dotnet-install.sh --channel 10.0
else:
    print('Reusing the installed .NET SDK.', flush=True)

# Install Cloudflared
if shutil.which('cloudflared') is None:
    !wget https://github.com/cloudflare/cloudflared/releases/download/2024.8.2/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb

# Keep the application local, and reuse it when this cell is repeated.
app = Path(APP_ROOT)
if app.exists() and not (app / '.git').is_dir():
    raise RuntimeError(f'Existing local path is not a SwarmUI checkout: {app}')
if not app.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, url, str(app)], check=True)
print(f'Local tools and checkout prepared in {time.monotonic() - tools_started:.1f} seconds.', flush=True)

In [ ]:
# 3. Prepare persistent storage and launch. Watch for a Cloudflare URL.
%cd $APP_ROOT

import runpy
import subprocess
import sys
from pathlib import Path

# Update the local program without modifying the old Drive installation.
subprocess.run(['git', 'fetch', '--depth', '1', url, branch], check=True)
subprocess.run(['git', 'reset', '--hard', 'FETCH_HEAD'], check=True)

# Protect this runtime's preinstalled packages before any Python installation.
# Use a fresh GPU runtime; restarting the Python session does not undo package changes.
runpy.run_path('colab/prepare-colab.py', run_name='__main__')

# Reuse existing Drive data and create only missing folders and local links.
storage_helpers = runpy.run_path('colab/prepare-storage.py')
run_logged = storage_helpers['run_logged']
storage = storage_helpers['prepare_storage'](Path.cwd(), DRIVE_ROOT, '/content/drive')
storage_helpers['prepare_backend'](Path.cwd())

# The normal launcher rebuilds only when required; keep temporary files local.
print('Starting SwarmUI. Server logs and the Cloudflare URL appear below.', flush=True)
try:
    run_logged(['bash', './launch-linux.sh', '--data_dir', str(storage['data']),
                '--launch_mode', 'none', '--cloudflared-path', 'cloudflared'],
               shutdown_timeout=30)
finally:
    print('Finishing pending Google Drive writes. Please wait before deleting the runtime.', flush=True)
    drive.flush_and_unmount(timeout_ms=60000)
    print('Google Drive synchronized and unmounted. Run the Drive mount cell before starting again.', flush=True)